In [1]:
import sys
sys.path.append('..')

import torch
from src.sae import SparseAutoencoder
from src.training import train_sae, train_multiple_seeds
from src.generators import generate_world_a, generate_world_b

X = generate_world_a(n_samples=1000, seed=42)
X_tensor = torch.tensor(X, dtype=torch.float32)

## Choosing the Sparsity Weight (λ)

Swept sparsity_weight over [0.01, 0.03, 0.05, 0.07, 0.1] on World A to find a reasonable tradeoff between reconstruction quality and sparsity. As expected, higher λ improves sparsity at the cost of reconstruction accuracy. Chose **λ = 0.03** as a middle ground: reconstruction loss stays low (~0.07) while sparsity loss drops meaningfully (~40%) from the λ=0.01 baseline.

In [2]:
for sw in [0.01, 0.03, 0.05, 0.07, 0.1]:
    m = SparseAutoencoder(input_dim=6, latent_dim=20)
    h = train_sae(m, X_tensor, n_epochs=200, sparsity_weight=sw)
    final_recon = h["recon"][-1]
    final_sparsity = h["sparsity"][-1]
    print(f"sparsity_weight={sw}: final_recon={final_recon:.4f}  final_sparsity={final_sparsity:.4f}")

Epoch 0: total=0.3558  recon=0.3341  sparsity=2.1739
Epoch 20: total=0.2993  recon=0.2785  sparsity=2.0845
Epoch 40: total=0.2554  recon=0.2333  sparsity=2.2024
Epoch 60: total=0.2173  recon=0.1924  sparsity=2.4875
Epoch 80: total=0.1830  recon=0.1541  sparsity=2.8910
Epoch 100: total=0.1556  recon=0.1229  sparsity=3.2677
Epoch 120: total=0.1336  recon=0.0981  sparsity=3.5438
Epoch 140: total=0.1158  recon=0.0785  sparsity=3.7291
Epoch 160: total=0.1015  recon=0.0626  sparsity=3.8917
Epoch 180: total=0.0898  recon=0.0495  sparsity=4.0293
sparsity_weight=0.01: final_recon=0.0396  final_sparsity=4.1249
Epoch 0: total=0.3802  recon=0.2787  sparsity=3.3860
Epoch 20: total=0.3233  recon=0.2281  sparsity=3.1757
Epoch 40: total=0.2838  recon=0.1912  sparsity=3.0870
Epoch 60: total=0.2545  recon=0.1633  sparsity=3.0408
Epoch 80: total=0.2303  recon=0.1414  sparsity=2.9653
Epoch 100: total=0.2084  recon=0.1222  sparsity=2.8748
Epoch 120: total=0.1890  recon=0.1043  sparsity=2.8226
Epoch 140: to

## Experiment 3: Training Across Multiple Seeds

Training 5 SAEs per world (same data, different random initializations via `torch.manual_seed(seed)` inside `train_multiple_seeds`) to check whether reconstruction and sparsity outcomes are consistent, or just a product of seed luck.

Note: an earlier ad-hoc comparison (outside this function) suggested World A and World B might need different λ values to match, this turned out to be a seeding artifact, since one world's model had been reseeded after a kernel restart and the other hadn't. `train_multiple_seeds` avoids this by explicitly seeding every model it creates, making all comparisons here reproducible.

In [3]:
from src.training import train_multiple_seeds
from src.generators import generate_world_a

results_a, X_tensor_a = train_multiple_seeds(generate_world_a, {}, n_seeds=5)

Epoch 0: total=0.3799  recon=0.3113  sparsity=2.2889
Epoch 20: total=0.3171  recon=0.2568  sparsity=2.0109
Epoch 40: total=0.2761  recon=0.2199  sparsity=1.8712
Epoch 60: total=0.2461  recon=0.1912  sparsity=1.8311
Epoch 80: total=0.2220  recon=0.1664  sparsity=1.8527
Epoch 100: total=0.2026  recon=0.1456  sparsity=1.8997
Epoch 120: total=0.1863  recon=0.1281  sparsity=1.9377
Epoch 140: total=0.1722  recon=0.1125  sparsity=1.9880
Epoch 160: total=0.1599  recon=0.0991  sparsity=2.0265
Epoch 180: total=0.1493  recon=0.0883  sparsity=2.0361
Seed 0: final_recon=0.0802  active_latents=6.60
Epoch 0: total=0.4016  recon=0.2870  sparsity=3.8214
Epoch 20: total=0.3389  recon=0.2320  sparsity=3.5634
Epoch 40: total=0.2966  recon=0.1950  sparsity=3.3835
Epoch 60: total=0.2649  recon=0.1675  sparsity=3.2450
Epoch 80: total=0.2385  recon=0.1445  sparsity=3.1332
Epoch 100: total=0.2161  recon=0.1251  sparsity=3.0332
Epoch 120: total=0.1961  recon=0.1073  sparsity=2.9613
Epoch 140: total=0.1787  reco

## World B - Same Procedure

Training 5 SAEs on World B using the identical procedure and λ=0.03, for direct seed-by-seed comparison against World A.

In [4]:
from src.training import train_multiple_seeds
from src.generators import generate_world_b
results_b, X_tensor_b = train_multiple_seeds(generate_world_b, {}, n_seeds=5)

Epoch 0: total=0.3899  recon=0.3201  sparsity=2.3256
Epoch 20: total=0.3251  recon=0.2639  sparsity=2.0421
Epoch 40: total=0.2827  recon=0.2258  sparsity=1.8962
Epoch 60: total=0.2516  recon=0.1960  sparsity=1.8539
Epoch 80: total=0.2269  recon=0.1703  sparsity=1.8897
Epoch 100: total=0.2077  recon=0.1492  sparsity=1.9482
Epoch 120: total=0.1920  recon=0.1322  sparsity=1.9931
Epoch 140: total=0.1781  recon=0.1172  sparsity=2.0297
Epoch 160: total=0.1656  recon=0.1040  sparsity=2.0536
Epoch 180: total=0.1546  recon=0.0931  sparsity=2.0525
Seed 0: final_recon=0.0845  active_latents=6.88
Epoch 0: total=0.4045  recon=0.2886  sparsity=3.8651
Epoch 20: total=0.3412  recon=0.2332  sparsity=3.6006
Epoch 40: total=0.2989  recon=0.1966  sparsity=3.4083
Epoch 60: total=0.2673  recon=0.1697  sparsity=3.2552
Epoch 80: total=0.2411  recon=0.1470  sparsity=3.1357
Epoch 100: total=0.2187  recon=0.1277  sparsity=3.0355
Epoch 120: total=0.1990  recon=0.1101  sparsity=2.9633
Epoch 140: total=0.1818  reco

## Result: Aggregate Statistics Match Closely

Comparing seed-by-seed, World A and World B show closely matched reconstruction loss and active-latent counts across all 5 seeds - differences between worlds are much smaller than the natural seed-to-seed variance within either world. 
This is consistent with **H0 (statistical hypothesis)**: no evidence yet that causal structure affects these aggregate training outcomes.

This doesn't resolve the core research question, though - these are coarse, aggregate statistics. Experiment 4 will check whether the *specific* latents learned actually correspond to A/B the same way, and whether the internal representation geometry differs between worlds.